# 🔗 Workflow Integration - Complete SAR Processing System

Welcome to Phase 4 of the Financial Services Agentic AI Project!

In this notebook, you'll integrate both AI agents into a complete **end-to-end SAR processing workflow** that demonstrates real-world financial compliance automation.

## 🎯 Learning Objectives
- Build a complete two-stage AI workflow with human oversight
- Implement human-in-the-loop decision gates for compliance
- Generate complete SAR documents from AI analysis
- Create comprehensive audit trails for regulatory examination
- Demonstrate cost optimization through intelligent agent coordination

## 📋 Business Context
This workflow simulates how banks actually process suspicious activity reports:
1. **Risk Screening**: AI agents analyze transaction patterns for suspicious activity
2. **Human Review**: Compliance officers review AI findings before proceeding
3. **Narrative Generation**: Only approved cases get full compliance documentation
4. **SAR Filing**: Complete regulatory forms are generated for submission
5. **Audit Documentation**: Every decision is logged for regulatory examination

## 🏗️ System Architecture

```
📊 CSV Data → 🔍 Risk Analyst → 👤 Human Decision → ✅ Compliance Officer → 📄 SAR Document
              (Chain-of-Thought)    (Gate)         (ReACT Framework)     (FinCEN Ready)
```

## 🚀 Prerequisites Check

Before starting, ensure you have completed:
- ✅ Phase 1: Foundation components (`foundation_sar.py`)
- ✅ Phase 2: Risk Analyst Agent (`risk_analyst_agent.py`)
- ✅ Phase 3: Compliance Officer Agent (`compliance_officer_agent.py`)
- ✅ Both agents pass their comprehensive test scenarios

If any component is missing, return to previous notebooks to complete implementation.

In [ ]:
# Setup and Environment Configuration
import os
import sys
import json
import pandas as pd
import uuid
import hashlib
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('../src'))

# Load environment variables
load_dotenv('../.env')

print("📚 Libraries imported successfully!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path")

📚 Libraries imported successfully!
🔐 Environment variables loaded
📂 Source directory added to Python path


In [ ]:
# OpenAI Setup for Vocareum
import openai

# Initialize OpenAI client for Vocareum
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your Vocareum OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        base_url="https://openai.vocareum.com/v1",
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized with Vocareum routing")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    print("📍 Base URL: https://openai.vocareum.com/v1")

✅ OpenAI client initialized with Vocareum routing
🔑 API key: voc-4949...3628
📍 Base URL: https://openai.vocareum.com/v1


In [ ]:

from foundation_sar import (
     CustomerData,
     AccountData,
     TransactionData,
     CaseData,
     RiskAnalystOutput,
     ComplianceOfficerOutput,
     ExplainabilityLogger,
     DataLoader,
     load_csv_data
 )
from risk_analyst_agent import RiskAnalystAgent
from compliance_officer_agent import ComplianceOfficerAgent


explainability_logger = ExplainabilityLogger("../outputs/audit_logs/workflow_integration.jsonl")
risk_agent = RiskAnalystAgent(client, explainability_logger)
compliance_agent = ComplianceOfficerAgent(client, explainability_logger)

print("✅ Ready to import components after implementation")

📋 TODO: Import your implemented components
Uncomment and modify these imports once you've implemented all components:
✅ Ready to import components after implementation


## 📊 Step 1: Data Loading and Preprocessing

Load the financial data and prepare it for analysis.

In [ ]:
# Step 1: Data Loading and Preprocessing

def load_and_preprocess_data():
    """
    Load CSV data and prepare for analysis.
    Applies dtype coercion and NaN handling consistent with foundation_sar.load_csv_data.
    """
    print("📊 Loading Financial Data...")

    customers_df    = pd.read_csv("../data/customers.csv", dtype={'ssn_last_4': str})
    accounts_df     = pd.read_csv("../data/accounts.csv")
    transactions_df = pd.read_csv("../data/transactions.csv")

    # Handle NaN in optional fields
    transactions_df['counterparty'] = transactions_df['counterparty'].fillna('')
    transactions_df['location']     = transactions_df['location'].fillna('')
    customers_df['phone']           = customers_df['phone'].fillna('')

    customers_data    = customers_df.to_dict('records')
    accounts_data     = accounts_df.to_dict('records')
    transactions_data = transactions_df.to_dict('records')

    print(f"📈 Loaded: {len(customers_data)} customers, "
          f"{len(accounts_data)} accounts, {len(transactions_data)} transactions")
    return customers_data, accounts_data, transactions_data

# Load data
customers_data, accounts_data, transactions_data = load_and_preprocess_data()

In [ ]:
customers_data[:2]

[{'account_id': 'CUST_0001_ACC_1',
  'customer_id': 'CUST_0001',
  'account_type': 'Checking',
  'opening_date': '2020-05-06',
  'current_balance': 51690.75,
  'average_monthly_balance': 0.0,
  'status': 'Closed'},
 {'account_id': 'CUST_0002_ACC_1',
  'customer_id': 'CUST_0002',
  'account_type': 'Money_Market',
  'opening_date': '2023-12-03',
  'current_balance': 92121.66,
  'average_monthly_balance': 172242.96,
  'status': 'Active'}]

## 🎯 Step 2: Customer Risk Screening

Implement intelligent customer screening to identify high-risk cases for detailed analysis.

In [ ]:
# Customer Risk Screening

def screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=5):
    """
    Risk-based customer screening.

    Criteria:
    - High risk rating (Medium or High)
    - Large transaction volume (>$100K total)
    - High transaction frequency (>50 transactions)

    Customers matching 2+ criteria are selected; results sorted by
    number of risk flags then total amount, returning top N.
    """
    selected_customers = []

    for customer in customers_data:
        customer_accounts = [
            acc for acc in accounts_data
            if acc['customer_id'] == customer['customer_id']
        ]
        customer_transactions = [
            txn for txn in transactions_data
            if any(txn['account_id'] == acc['account_id'] for acc in customer_accounts)
        ]

        total_amount      = sum(abs(txn['amount']) for txn in customer_transactions)
        transaction_count = len(customer_transactions)
        risk_rating       = customer['risk_rating']

        risk_flags = []
        if risk_rating in ['Medium', 'High']:
            risk_flags.append('high_risk_rating')
        if total_amount > 100000:
            risk_flags.append('large_amounts')
        if transaction_count > 50:
            risk_flags.append('high_frequency')

        if len(risk_flags) >= 2:
            selected_customers.append({
                'customer':          customer,
                'accounts':          customer_accounts,
                'transactions':      customer_transactions,
                'total_amount':      total_amount,
                'transaction_count': transaction_count,
                'risk_flags':        risk_flags
            })

    # Sort and slice OUTSIDE the loop
    selected_customers.sort(
        key=lambda x: (len(x['risk_flags']), x['total_amount']),
        reverse=True
    )
    result = selected_customers[:top_n]

    print(f"🔍 Screened {len(result)} high-risk customers from {len(customers_data)} total:")
    for c in result:
        print(f"   • {c['customer']['name']} | flags: {c['risk_flags']} | "
              f"txns: {c['transaction_count']} | total: ${c['total_amount']:,.0f}")
    return result

# Run customer screening
selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data)

## 🤖 Step 3: Two-Stage AI Analysis with Human Gates

Implement the core two-stage workflow:
1. **Stage 1**: Risk Analyst performs Chain-of-Thought analysis
2. **Human Gate**: Review and decision to proceed
3. **Stage 2**: Compliance Officer generates ReACT narratives (only if approved)

In [ ]:
# Two-Stage AI Workflow with Human Decision Gates

def run_two_stage_sar_workflow(selected_customers):
    """
    Complete two-stage SAR processing workflow.

    For each customer:
    1. Create CaseData object
    2. Run Risk Analyst analysis (Chain-of-Thought)
    3. Present findings to human reviewer
    4. Get human decision (proceed/reject)
    5. If approved: Run Compliance Officer (ReACT)
    6. Generate and save complete SAR document
    7. Log all decisions for audit
    """
    print("🤖 Two-Stage SAR Processing Workflow")

    processed_cases = []
    approved_sars   = []
    rejected_cases  = []
    audit_decisions = []

    for i, customer_data in enumerate(selected_customers, 1):
        print(f"\n🔍 CUSTOMER {i}/{len(selected_customers)}: {customer_data['customer']['name']}")
        print("=" * 60)

        try:
            loader    = DataLoader(explainability_logger)
            case_data = loader.create_case_from_data(
                customer_data['customer'],
                customer_data['accounts'],
                customer_data['transactions']
            )

            # STAGE 1: Risk Analysis
            print("🔍 STAGE 1: Risk Analysis")
            risk_analysis = risk_agent.analyze_case(case_data)
            print(f"   Classification: {risk_analysis.classification}")
            print(f"   Confidence:     {risk_analysis.confidence_score:.2f}")
            print(f"   Risk Level:     {risk_analysis.risk_level}")
            print(f"   Reasoning:      {risk_analysis.reasoning[:120]}...")

            processed_cases.append({
                'case_id':        case_data.case_id,
                'customer_name':  case_data.customer.name,
                'classification': risk_analysis.classification,
                'risk_level':     risk_analysis.risk_level,
            })

            # HUMAN DECISION GATE
            decision       = input("\n🤔 Proceed with SAR filing? (yes/no): ").strip().lower()
            should_proceed = decision in ['yes', 'y']

            if should_proceed:
                # STAGE 2: Compliance Narrative
                print("\n📝 STAGE 2: Compliance Narrative Generation")
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)

                sar_document = create_sar_document(case_data, risk_analysis, compliance_review)
                save_sar_document(sar_document)
                approved_sars.append(sar_document)
                print(f"✅ SAR FILED: {sar_document['sar_metadata']['sar_id']}")
            else:
                rejected_cases.append({'case_id': case_data.case_id, 'reason': 'human_rejection'})
                print("❌ SAR REJECTED by human reviewer")

            audit_decisions.append({
                'case_id':          case_data.case_id,
                'customer_name':    case_data.customer.name,
                'decision':         'PROCEED' if should_proceed else 'REJECT',
                'ai_classification': risk_analysis.classification,
                'ai_confidence':    risk_analysis.confidence_score,
                'reviewer_decision': decision
            })

        except Exception as e:
            print(f"❌ Error processing {customer_data['customer']['name']}: {e}")

    return processed_cases, approved_sars, rejected_cases, audit_decisions

# Run the complete workflow
processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(selected_customers)

## 📄 Step 4: SAR Document Generation

Create complete, FinCEN-ready SAR documents with all required metadata.

In [ ]:
# Students: Create complete SAR documents for regulatory submission

def create_sar_document(case_data, risk_analysis, compliance_review):
    """
    Create complete SAR document
    
    SAR document should include:
    1. SAR metadata (ID, filing date, type, checksum)
    2. Subject information (customer details)
    3. Suspicious activity description
    4. AI analysis results
    5. Compliance narrative
    6. Regulatory citations
    7. Filing institution information
    """
    print("📄 Creating SAR Document")
    print("📋 Generating unique SAR ID")
    print("📋 Including all required metadata")
    print("📋 Formatting for FinCEN submission")
    
   
    sar_id = f"SAR_{uuid.uuid4()}"
    filing_date = datetime.now().isoformat()
     
    sar_document = {
         'sar_metadata': {
         'sar_id': sar_id,
          'filing_date': filing_date,
             'filing_type': 'Suspicious Activity Report',
             'ai_generated': True,
             'review_status': 'human_approved'
         },
         'subject_information': {
             'customer_name': case_data.customer.name,
             'customer_id': case_data.customer.customer_id,
             'address': case_data.customer.address,
             'customer_since': case_data.customer.customer_since,
             'risk_rating': case_data.customer.risk_rating
         },
         'suspicious_activity': {
             'classification': risk_analysis.classification,
             'risk_level': risk_analysis.risk_level,
             'confidence_score': risk_analysis.confidence_score,
             'narrative': compliance_review.narrative,
             'key_indicators': risk_analysis.key_indicators,
             'ai_reasoning': risk_analysis.reasoning
         },
         'regulatory_compliance': {
             'citations': getattr(compliance_review, 'regulatory_citations', []),
             'narrative_word_count': len(compliance_review.narrative.split()),
             'compliance_status': 'approved'
         },
        'audit_trail': {
             'case_id': case_data.case_id,
             'processing_date': filing_date,
             'ai_agents_used': ['RiskAnalyst', 'ComplianceOfficer'],
             'human_reviewer': 'compliance_officer'
         }
     }
     
    return sar_document
    
   

def save_sar_document(sar_document):
    """Save SAR document to outputs directory"""
    print("📋 Saving SAR to ../outputs/filed_sars/ directory")
    os.makedirs("../outputs/filed_sars", exist_ok=True)
    filename = f"../outputs/filed_sars/{sar_document['sar_metadata']['sar_id']}.json"
    with open(filename, 'w') as f:
        json.dump(sar_document, f, indent=2)

print("📄 SAR document generation functions defined")

## 📊 Step 5: Workflow Metrics and Analysis

Analyze the efficiency and effectiveness of your AI-powered SAR processing system.

In [ ]:
# Step 5: Workflow Metrics and Analysis

def analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions):
    """Calculate workflow efficiency and cost optimization metrics."""
    print("📊 Workflow Efficiency Analysis")

    total_cases        = len(processed_cases)
    approved_count     = len(approved_sars)
    rejected_count     = len(rejected_cases)
    approval_rate      = approved_count / total_cases if total_cases > 0 else 0
    rejection_rate     = rejected_count / total_cases if total_cases > 0 else 0

    print(f"\n📈 WORKFLOW METRICS:")
    print(f"   Total Cases Processed: {total_cases}")
    print(f"   SARs Filed:            {approved_count}")
    print(f"   Cases Rejected:        {rejected_count}")
    print(f"   Approval Rate:         {approval_rate:.1%}")
    print(f"   Rejection Rate:        {rejection_rate:.1%}")

    print(f"\n💰 COST OPTIMIZATION:")
    print(f"   Two-stage processing avoids running the Compliance Officer")
    print(f"   on rejected cases — {rejection_rate:.1%} of Stage 2 API calls saved.")
    if total_cases > 0:
        print(f"   ({rejected_count} of {total_cases} cases never reached narrative generation)")


def validate_ai_decisions(audit_decisions):
    """Analyse AI decision patterns from the audit trail."""
    if not audit_decisions:
        print("⚠️  No audit decisions to analyse.")
        return

    print("\n🔍 AI DECISION ANALYSIS")
    print("=" * 50)

    # --- Classification breakdown ---
    from collections import Counter
    classifications = Counter(d['ai_classification'] for d in audit_decisions)
    print("\n📊 Classification Breakdown:")
    for cls, count in classifications.most_common():
        print(f"   {cls:<20} {count} case{'s' if count != 1 else ''}")

    # --- Confidence score distribution ---
    scores = [d['ai_confidence'] for d in audit_decisions]
    print(f"\n📉 Confidence Scores:")
    print(f"   Min:     {min(scores):.2f}")
    print(f"   Max:     {max(scores):.2f}")
    print(f"   Average: {sum(scores)/len(scores):.2f}")

    high_conf   = [s for s in scores if s >= 0.8]
    medium_conf = [s for s in scores if 0.5 <= s < 0.8]
    low_conf    = [s for s in scores if s < 0.5]
    print(f"   High (≥0.8):      {len(high_conf)} cases")
    print(f"   Medium (0.5–0.8): {len(medium_conf)} cases")
    print(f"   Low (<0.5):       {len(low_conf)} cases")

    # --- Human decision patterns ---
    proceeded = [d for d in audit_decisions if d['decision'] == 'PROCEED']
    rejected  = [d for d in audit_decisions if d['decision'] == 'REJECT']
    print(f"\n👤 Human Decision Patterns:")
    print(f"   Approved: {len(proceeded)} | Rejected: {len(rejected)}")

    # Override rate — human rejected despite AI flagging (all cases are AI-flagged here)
    if rejected:
        avg_conf_rejected = sum(d['ai_confidence'] for d in rejected) / len(rejected)
        print(f"   Avg confidence on rejected cases: {avg_conf_rejected:.2f}")
        print(f"   Classifications rejected by reviewer:")
        rejected_cls = Counter(d['ai_classification'] for d in rejected)
        for cls, count in rejected_cls.most_common():
            print(f"     • {cls}: {count}")

    # --- Per-case summary ---
    print(f"\n📋 Case-by-Case Summary:")
    for d in audit_decisions:
        icon = "✅" if d['decision'] == 'PROCEED' else "❌"
        print(f"   {icon} {d['customer_name']:<25} "
              f"{d['ai_classification']:<20} "
              f"conf: {d['ai_confidence']:.2f}  →  {d['decision']}")


# Run analysis
analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)
validate_ai_decisions(audit_decisions)

## 🏁 Step 6: Complete System Demonstration

Test your complete system with comprehensive scenarios to validate production readiness.

In [ ]:
# Complete System Demonstration

def demonstrate_complete_system():
    """
    Run complete system demonstration.
    
    1. Load fresh data from CSVs
    2. Screen top 3 high-risk customers
    3. Run full two-stage workflow with human gates
    4. Print efficiency metrics
    5. Report output locations
    """
    print("🚀 Running complete system test...")

    # Load fresh data
    customers_data, accounts_data, transactions_data = load_and_preprocess_data()

    # Screen customers
    selected_customers = screen_high_risk_customers(
        customers_data, accounts_data, transactions_data, top_n=3
    )

    # Run workflow
    processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(selected_customers)

    # Generate final report
    analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)

    print(f"\n🎉 System demonstration complete!")
    print(f"📄 SAR documents saved to: ../outputs/filed_sars/")
    print(f"📊 Audit logs saved to: ../outputs/audit_logs/")

demonstrate_complete_system()

## 📝 Implementation Checklist

### ✅ Workflow Integration Deliverables
- [ ] **Data Loading**: Load and preprocess CSV data with proper error handling
- [ ] **Customer Screening**: Implement risk-based screening to identify high-risk cases
- [ ] **Two-Stage Workflow**: Build complete Risk Analyst → Human Gate → Compliance Officer flow
- [ ] **Human Decision Gates**: Implement interactive approval/rejection points
- [ ] **SAR Document Generation**: Create complete FinCEN-ready documents with metadata
- [ ] **Audit Trail Creation**: Log all decisions and reasoning for regulatory examination
- [ ] **Efficiency Metrics**: Calculate cost optimization and processing efficiency
- [ ] **System Demonstration**: Test complete workflow with multiple scenarios

### ✅ Testing and Validation Requirements
- [ ] **Component Validation**: Verify all foundation components and agents are available
- [ ] **Integration Testing**: Run comprehensive test suites for all components with proper sys.path setup
- [ ] **End-to-End Testing**: Test complete workflow with automated scenarios
- [ ] **Error Handling Testing**: Validate graceful handling of edge cases and failures
- [ ] **Output Validation**: Ensure SAR documents meet regulatory standards
- [ ] **Performance Testing**: Measure workflow efficiency and processing times

### ✅ Technical Requirements
- [ ] **Error Handling**: Robust exception handling for all workflow steps
- [ ] **Data Validation**: Proper validation of all inputs and outputs
- [ ] **File Management**: Organize outputs in appropriate directories
- [ ] **Logging**: Comprehensive audit logging for compliance
- [ ] **Performance**: Efficient processing of multiple cases
- [ ] **User Experience**: Clear prompts and feedback for human reviewers
- [ ] **Test Infrastructure**: Proper test imports and sys.path configuration

### ✅ Business Requirements  
- [ ] **Regulatory Compliance**: Ensure all SAR documents meet FinCEN requirements
- [ ] **Cost Optimization**: Demonstrate savings from two-stage processing
- [ ] **Audit Readiness**: Create examination-ready documentation
- [ ] **Quality Assurance**: Validate AI decisions with human oversight
- [ ] **Scalability**: Design for processing larger datasets
- [ ] **Production Readiness**: Complete testing validates system reliability

## 🎯 Success Criteria

By completion, your integrated system should:
- ✅ Process real financial data with proper validation
- ✅ Execute complete two-stage AI workflow with human gates
- ✅ Generate regulatory-compliant SAR documents
- ✅ Create comprehensive audit trails for all decisions
- ✅ Demonstrate measurable cost optimization benefits
- ✅ Handle errors gracefully and provide clear user feedback
- ✅ Pass all integration and end-to-end tests
- ✅ Meet production-ready quality standards

## 🚀 Next Steps

1. **Complete Implementation**: Fill in all TODO sections with working code
2. **Run Integration Tests**: Validate all components work together properly
3. **Execute End-to-End Tests**: Test complete workflow with automated scenarios
4. **Test Thoroughly**: Run complete workflow with various manual scenarios
5. **Validate Outputs**: Ensure SAR documents meet regulatory requirements
6. **Document Results**: Create final project documentation and metrics
7. **Prepare Presentation**: Demonstrate your system's capabilities and business value

**Congratulations on building a complete AI-powered SAR processing system! 🎉**

## 🧪 Step 7: Workflow Testing and Validation

Before finalizing your implementation, validate your complete system with comprehensive testing.

In [ ]:
# 🧪 Workflow Integration Testing
# Validate your complete system with integration tests

import sys
import os

# Add tests directory to Python path for importing test modules
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Added tests directory to Python path: {tests_path}")

def run_integration_tests():
    """
    Run comprehensive integration tests to validate the complete workflow
    """
    print("🧪 Comprehensive Integration Testing")
    
    try:
        import pytest

        print("🔍 Running Foundation Component Tests...")
        foundation_result = pytest.main([
            f"{tests_path}/test_foundation.py",
            "-v",
            "--tb=short",
            "-q"
        ])

        print("🔍 Running Risk Analyst Agent Tests...")
        risk_result = pytest.main([
            f"{tests_path}/test_risk_analyst.py",
            "-v",
            "--tb=short",
            "-q"
        ])

        print("📝 Running Compliance Officer Agent Tests...")
        compliance_result = pytest.main([
            f"{tests_path}/test_compliance_officer.py",
            "-v",
            "--tb=short",
            "-q"
        ])

        all_passed = foundation_result == 0 and risk_result == 0 and compliance_result == 0

        print("\n" + "="*60)
        print("📊 INTEGRATION TEST RESULTS:")
        print(f"   Foundation Components: {'✅ PASS' if foundation_result == 0 else '❌ FAIL'}")
        print(f"   Risk Analyst Agent:    {'✅ PASS' if risk_result == 0 else '❌ FAIL'}")
        print(f"   Compliance Officer:    {'✅ PASS' if compliance_result == 0 else '❌ FAIL'}")
        print(f"   Overall Status:        {'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED'}")

        if all_passed:
            print("\n🎉 Your system is ready for production workflow testing!")
        else:
            print("\n⚠️ Fix failing tests before running the complete workflow.")

        return all_passed

    except ImportError as e:
        print(f"❌ Import Error: {e}")
        return False

def validate_workflow_components():
    """Validate that all required components are available for integration"""
    print("🔍 Validating Workflow Components")

    components_status = {
        'foundation_sar': False,
        'risk_analyst_agent': False,
        'compliance_officer_agent': False,
        'test_modules': False
    }

    try:
        from foundation_sar import CustomerData, CaseData, ExplainabilityLogger, DataLoader
        components_status['foundation_sar'] = True
        print("✅ Foundation components available")
    except ImportError:
        print("❌ Foundation components not available")

    try:
        from risk_analyst_agent import RiskAnalystAgent
        components_status['risk_analyst_agent'] = True
        print("✅ Risk Analyst Agent available")
    except ImportError:
        print("❌ Risk Analyst Agent not available")

    try:
        from compliance_officer_agent import ComplianceOfficerAgent
        components_status['compliance_officer_agent'] = True
        print("✅ Compliance Officer Agent available")
    except ImportError:
        print("❌ Compliance Officer Agent not available")

    try:
        from test_foundation import TestCustomerData
        from test_risk_analyst import TestRiskAnalystAgent
        from test_compliance_officer import TestComplianceOfficerAgent
        components_status['test_modules'] = True
        print("✅ Test modules available")
    except ImportError:
        print("❌ Test modules not available")

    all_ready = all(components_status.values())

    print(f"\n📊 Component Status: {'✅ ALL READY' if all_ready else '⚠️ INCOMPLETE'}")
    if not all_ready:
        print("💡 Complete missing components before running integration tests")

    return all_ready

# Run component validation
components_ready = validate_workflow_components()

# Run integration tests if components are ready
if components_ready:
    print("\n🚀 All components ready - running integration tests!")
    run_integration_tests()
else:
    print("\n📋 Complete component implementation first, then run integration tests")

In [ ]:
# 🎯 End-to-End Workflow Testing
# Test the complete workflow with known test scenarios (automated decisions)

def test_complete_workflow():
    """
    Test the complete SAR processing workflow end-to-end with automated decisions.
    Uses simulated human approvals so the test runs without interactive input.
    """
    print("🎯 End-to-End Workflow Testing")

    try:
        print("🚀 Starting end-to-end workflow test...")

        # Test data preparation
        print("📊 Loading test data...")
        customers_data, accounts_data, transactions_data = load_and_preprocess_data()

        if not customers_data:
            print("⚠️ No test data available - implement data loading first")
            return False

        # Test customer screening
        print("🔍 Testing customer screening...")
        selected_customers = screen_high_risk_customers(
            customers_data, accounts_data, transactions_data, top_n=2
        )

        if not selected_customers:
            print("⚠️ No customers selected - check screening criteria")
            return False

        print(f"✅ Selected {len(selected_customers)} customers for testing")

        # Test workflow with automated decisions (simulate approval for all cases)
        print("🤖 Testing automated workflow (simulating human approval)...")
        test_results = {
            'cases_processed': 0,
            'sars_generated': 0,
            'errors': []
        }

        for customer_data in selected_customers:
            try:
                # Create case
                loader = DataLoader(explainability_logger)
                case_data = loader.create_case_from_data(
                    customer_data['customer'],
                    customer_data['accounts'],
                    customer_data['transactions']
                )

                # Test Risk Analyst
                risk_analysis = risk_agent.analyze_case(case_data)
                assert hasattr(risk_analysis, 'classification'), "Risk analysis missing classification"
                assert hasattr(risk_analysis, 'confidence_score'), "Risk analysis missing confidence"

                # Test Compliance Officer (simulate approval)
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
                assert hasattr(compliance_review, 'narrative'), "Compliance review missing narrative"

                # Test SAR generation
                sar_document = create_sar_document(case_data, risk_analysis, compliance_review)
                assert sar_document, "SAR document generation failed"

                # Save SAR
                save_sar_document(sar_document)

                test_results['cases_processed'] += 1
                test_results['sars_generated'] += 1

                print(f"✅ Successfully processed: {customer_data['customer']['name']}")
                print(f"   Classification: {risk_analysis.classification} | Confidence: {risk_analysis.confidence_score:.2f}")
                print(f"   Narrative ({len(compliance_review.narrative.split())} words): {compliance_review.narrative[:80]}...")

            except Exception as e:
                test_results['errors'].append(f"Error processing {customer_data['customer']['name']}: {e}")
                print(f"❌ Error: {customer_data['customer']['name']}: {e}")

        # Test results summary
        print("\n" + "="*60)
        print("📊 END-TO-END TEST RESULTS:")
        print(f"   Cases Processed: {test_results['cases_processed']}")
        print(f"   SARs Generated:  {test_results['sars_generated']}")
        print(f"   Errors:          {len(test_results['errors'])}")

        if test_results['errors']:
            print("❌ Test Errors:")
            for error in test_results['errors']:
                print(f"     • {error}")

        success = len(test_results['errors']) == 0 and test_results['cases_processed'] > 0

        if success:
            print("\n🎉 END-TO-END TEST PASSED!")
            print("✅ Your complete workflow is ready for production use!")
        else:
            print("\n⚠️ END-TO-END TEST HAD ISSUES")
            print("📝 Fix the errors above before deploying to production")

        return success

    except Exception as e:
        print(f"❌ End-to-end test failed: {e}")
        import traceback
        traceback.print_exc()
        return False

# Run end-to-end test
test_success = test_complete_workflow()